# Global Shipping & Port Congestion Analytics

## Objetivo

Analizar el tráfico portuario internacional y su relación con variables económicas y climáticas utilizando una arquitectura Medallion en Databricks.

## Business Problem

Analizar la evolución del tráfico portuario mundial y construir una plataforma analítica utilizando una arquitectura Medallion (Bronze, Silver y Gold) sobre Databricks

## Fuente de datos

- World Bank API
- NOAA API

## Arquitectura

Bronze → Silver → Gold


# 1. Ingesta de Datos

Objetivo:

Através de APIs, conectar la base de datos del Banco Mundial al notebook para la consumir información.

In [0]:

import requests
url = "https://api.worldbank.org/v2/country/all/indicator/IS.SHP.GOOD.TU?format=json"


# 2. Capa Bronze

Objetivo:

Almacenar los datos originales obtenidos de la API, sin realizar ninguna transformación previa. 

In [0]:
response = requests.get(url)

##2.1. Extracción y Exploración Inicial

Validación de estructura JSON para identificar tipo de datos, cantidad de información, etc.

### Objetivo: 
Contar con una estructura de DataFrame con los registros de todas las páginas.

In [0]:
data = response.json()
print(type(data))
print(len(data))
print(data[0])

<class 'list'>
2
{'page': 1, 'pages': 350, 'per_page': 50, 'total': 17490, 'sourceid': '2', 'lastupdated': '2026-07-13'}


La API World Bank nos devolvió una estructura de datos de tipo lista, "metadata, data" ya que data[0] contiene los metadatos: 
{
'page': 1,
'pages': 350,
'per_page': 50,
'total': 17490,
'sourceid': '2',
'lastupdated': '2026-07-13'
}

Para continuar el análisis exploratorio de los registros, inspeccionaremos el primer registro de la estructura

In [0]:
print(type(data[1]))
print(len(data[1]))
print(data[1][0])

<class 'list'>
50
{'indicator': {'id': 'IS.SHP.GOOD.TU', 'value': 'Container port traffic (TEU: 20 foot equivalent units)'}, 'country': {'id': 'ZH', 'value': 'Africa Eastern and Southern'}, 'countryiso3code': 'AFE', 'date': '2025', 'value': None, 'unit': '', 'obs_status': '', 'decimal': 0}


En esta primera inspección podemos identificar que los registros cuentan con los siguientes campos:'indicator', 'country', 'countryiso3code', 'value', 'unit', 'obs_status', 'decimal' . Sin embargo, también observamos que al menos 2 de los primeros campos, contienen datos anidados; por lo que, para construir el DataFrame de la capa Bronze usaremos la librería de pandas para evitar los problemas que spark pueda tener infiriendo el esquema automáticamente. 

In [0]:
import pandas as pd

records = data[1]
pdf = pd.DataFrame(records)
df = spark.createDataFrame(pdf)
display(df)

indicator,country,countryiso3code,date,value,unit,obs_status,decimal
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2025,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2024,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2023,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2022,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2021,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2020,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2019,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2018,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2017,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2016,null,,,0


Con el resultado obtenido, podemos afirmar que con el código ejecutado, logramos convertir los datos de la AI en un DataFrame; sin embargo, también podemos identificar algunos errores: 

### A. Los metadatos no coinciden con la cantidad de registros guardados. 

La cantidad de filas muestra que solo logramos cargar los datos correspondientes a la primera página (50 filas de las 17490) de todas las que se identificaron durante la exploración de los registros. También es importante considerar que los campos 'indicator' y 'country' permanecen anidados.

### B. Campos con estructuras anidadas

Los siguientes campos llegaron como estructuras anidadas:

indicator = {"id": "...", "value": "..."}
country = {"id": "...", "value": "..."}

Sin embargo, dado que aun nos encontramos en la capa bronze, aun no desanidaremos ya que en esta capa interesa conservar la información con la estructura más cercana a la fuente.

### c. Valores nulos en el campo 'value'

Todos los registros del campo value, muestran valores nulos. Es decir, en lugar de mostrar países y puertos individuales; en el primer bloque cargado, solo se muestrna valores nulos.

value = {null}

### d. Valores repetidos en los campos 'countryiso3code' y 'country.value'

countryiso3code = AFE
country.value = Africa Eastern and Southern


[](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api-documentation)

También podemos identificar que la API de indicadores del Banco Mundial utiliza el primer elemento de la respuesta como metadata y el segundo como colección de registros. Además, admite parámetros de paginación en sus consultas.

_Considerando las primeras observaciones que identificamos en el primer DataFrame generado para la capa bronze, realizaremos algunas correcciones para lograr cargar todos los registros y no solo los que se encuentran en la primera página._

In [0]:
import requests
from datetime import datetime

indicator_code = "IS.SHP.GOOD.TU"
base_url = (
    f"https://api.worldbank.org/v2/country/all/"
    f"indicator/{indicator_code}"
)

params = {
    "format": "json",
    "per_page": 1000,
    "page": 1
}

all_records = []
total_pages = 1

while params["page"] <= total_pages:

    response = requests.get(
        base_url,
        params=params,
        timeout=60
    )

    response.raise_for_status()
    page_data = response.json()

    if not page_data or len(page_data) < 2:
        break
    metadata = page_data[0]
    records_page = page_data[1] or []
    
    total_pages = int(metadata["pages"])
    all_records.extend(records_page)

    print(
        f"Página {params['page']} de {total_pages} "
        f"| Registros acumulados: {len(all_records)}"
    )

    params["page"] += 1

Página 1 de 18 | Registros acumulados: 1000
Página 2 de 18 | Registros acumulados: 2000
Página 3 de 18 | Registros acumulados: 3000
Página 4 de 18 | Registros acumulados: 4000
Página 5 de 18 | Registros acumulados: 5000
Página 6 de 18 | Registros acumulados: 6000
Página 7 de 18 | Registros acumulados: 7000
Página 8 de 18 | Registros acumulados: 8000
Página 9 de 18 | Registros acumulados: 9000
Página 10 de 18 | Registros acumulados: 10000
Página 11 de 18 | Registros acumulados: 11000
Página 12 de 18 | Registros acumulados: 12000
Página 13 de 18 | Registros acumulados: 13000
Página 14 de 18 | Registros acumulados: 14000
Página 15 de 18 | Registros acumulados: 15000
Página 16 de 18 | Registros acumulados: 16000
Página 17 de 18 | Registros acumulados: 17000
Página 18 de 18 | Registros acumulados: 17490


Para verificar que el código haya generado el DataFrame con todos los registros, revisamos que el "len" del dataframe, contenga la misma cantidad de datos que se detallan en los metadatos

In [0]:
print(f"Total extraído: {len(all_records)}")
print(all_records[0])

Total extraído: 17490
{'indicator': {'id': 'IS.SHP.GOOD.TU', 'value': 'Container port traffic (TEU: 20 foot equivalent units)'}, 'country': {'id': 'ZH', 'value': 'Africa Eastern and Southern'}, 'countryiso3code': 'AFE', 'date': '2025', 'value': None, 'unit': '', 'obs_status': '', 'decimal': 0}


Habiendo realizado las verificaciones correspondientes de los registros, confirmamos que finalizamos la etapa de extración de datos de la fuente. 'all_records'

### 2.2. Construcción del DataFrame Bronze

En este apartado, crearemos el DataFrame bronze con Pandas. La mejor opción, sería crear el dataframe serializado en Json, con el objetivo de que Spark interprete los objetos anidados como estructuras. Sin embargo, dado que estamos trabajando con Serverless,SparkContext no es soportado

### Objetivo: 
Convertir los datos obtenidos en un DataFrame Spark.

In [0]:
#Matenemos Bronze tal cual viene desde Pandas
pdf = pd.DataFrame(all_records)
df_bronze = spark.createDataFrame(pdf)

Para crear correctamente la capa bronze, verificamos los ctálogos y esquemas disponibles, así como los que utilizamos actualemnte.

In [0]:
#Averiguamos donde estamos trabajando
#Mostramos los catálogos existentes
spark.sql("SHOW CATALOGS").show(truncate=False)

+---------+
|catalog  |
+---------+
|samples  |
|system   |
|workspace|
+---------+



In [0]:
#Mostramos en cuál de los catálogos estamos trabajando
spark.sql("SELECT current_catalog()").show()

+-----------------+
|current_catalog()|
+-----------------+
|        workspace|
+-----------------+



In [0]:
#Mostramos cuales son los esquemas existentes
spark.sql("SHOW SCHEMAS").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|gold              |
|information_schema|
|silver            |
+------------------+



## Entorno Databricks Actual
Catálogo utilizado: Workspace
Schemas: Default

Considerando los resultados obtenidos, podemos observar que no existe el catálogo 'global_shipping' ni el schema 'bronze' ni la tabla 'world_bank_container_traffic'; por lo que, procedemos a crear nuestros propios esquemas.

In [0]:
#Paso1: Creamos el schema Bronze
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.bronze
""")

DataFrame[]

In [0]:
#Verificamos correcta creación del Schema
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|gold              |
|information_schema|
|silver            |
+------------------+



In [0]:
#Paso2: Guardar la tabla Bronze
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.bronze.world_bank_container_traffic"
    )

In [0]:
#Paso3: Validar correcta creación de la tabla 
spark.sql("""
SELECT *
FROM workspace.bronze.world_bank_container_traffic
LIMIT 10
""")

DataFrame[indicator: struct<id:string,value:string>, country: struct<id:string,value:string>, countryiso3code: string, date: string, value: double, unit: string, obs_status: string, decimal: bigint]

In [0]:
#Verificar correcto guardado
spark.sql("""
SHOW TABLES IN workspace.bronze
""").show(truncate=False)

+--------+----------------------------+-----------+
|database|tableName                   |isTemporary|
+--------+----------------------------+-----------+
|bronze  |world_bank_container_traffic|false      |
+--------+----------------------------+-----------+



In [0]:
#Validar correcta lectura desde Bronze
df_test = spark.table(
    "workspace.bronze.world_bank_container_traffic"

)

display(df_test.limit(10))

indicator,country,countryiso3code,date,value,unit,obs_status,decimal
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2017,5.6577838324E7,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2016,5.2672486324E7,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2015,5.2329947324E7,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2014,5.0087043324E7,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2013,4.8107682461E7,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2012,4.7010660461E7,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2011,4.5553546461E7,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2010,4.4941496461E7,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2009,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2008,null,,,0


## Paso adicional: Creación de esquemas de todas las capas
Para evitar el error de no encontrar los esquema de cada capa, creamos todas las capas de forma anticipada.

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.bronze
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.silver
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.gold
""")

DataFrame[]

# Arquitectura Medallion
Catalog:
workspace
Schemas:
- bronze
- silver
- gold

Fuente:
World Bank API

Proyecto:
Global Shipping & Port Congestion Analytics

#Validación de calidad de datos de capa bronze
Verificaremos que la cantidad de datos del DataFrame coincidan con la cantida de datos de la tabla spark, para verificar la calidad de los datos de la capa bronze.

In [0]:
#Conteo

df_bronze.count()

17490

In [0]:
#Verificación de tabla bronze tabla Spark

spark.table(
    "workspace.bronze.world_bank_container_traffic"
).count()

17490

In [0]:
#Mostrar tablas creadas
display(df_bronze)

indicator,country,countryiso3code,date,value,unit,obs_status,decimal
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2025,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2024,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2023,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2022,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2021,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2020,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2019,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2018,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2017,null,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZH, Africa Eastern and Southern)",AFE,2016,null,,,0


# Resumen Capa Bronze

Objetivo:

Persistir los datos obtenidos desde la World Bank API manteniendo la estructura original de la fuente.

Transformaciones aplicadas:

- Ninguna transformación de negocio.
- Conservación de campos anidados.
- Almacenamiento en formato Delta.

Tabla generada:

workspace.bronze.world_bank_container_traffic

# Data Profiling y validación de la calidad de datos
Objetivo:
Evaluar la calidad y estructura de los datos ingresados.

In [0]:
#Verificación de esquema de arquitectura Bronze
df_bronze.printSchema()
df_bronze.show(5)

root
 |-- indicator: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- value: string (nullable = true)
 |-- country: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- value: string (nullable = true)
 |-- countryiso3code: string (nullable = true)
 |-- date: string (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- obs_status: string (nullable = true)
 |-- decimal: long (nullable = true)

+--------------------+--------------------+---------------+----+-----+----+----------+-------+
|           indicator|             country|countryiso3code|date|value|unit|obs_status|decimal|
+--------------------+--------------------+---------------+----+-----+----+----------+-------+
|{IS.SHP.GOOD.TU, ...|{ZH, Africa Easte...|            AFE|2025| NULL|    |          |      0|
|{IS.SHP.GOOD.TU, ...|{ZH, Africa Easte...|            AFE|2024| NULL|    |          |      0|
|{IS.SHP.GOOD.TU, ...|{ZH, Africa Easte...

En el esquema, spark detectó que todos los valores de la columna "value" son nulos; por lo qué asignó tipo "void", en lugar de asignar tipo "double" o "string"; por lo que validaremos si efectivamente todos los valores del campo "value" son nulos o si solo se trata de una inferencia incorrecta de spark.

In [0]:
#Mostramos los primeros datos del campo "value" de la tabla bronze:
display(
    df_bronze.select(
        "country.value",
        "date",
        "value"
)
    .limit(20)
)

value,date,value
Africa Eastern and Southern,2025,null
Africa Eastern and Southern,2024,null
Africa Eastern and Southern,2023,null
Africa Eastern and Southern,2022,null
Africa Eastern and Southern,2021,null
Africa Eastern and Southern,2020,null
Africa Eastern and Southern,2019,null
Africa Eastern and Southern,2018,null
Africa Eastern and Southern,2017,null
Africa Eastern and Southern,2016,null


Mostrando los primeros registros del campo "value", identificamos que todos son nulos; sin embargo, debemos asegurarnos que efectivamente, todos los registros sean nulos y no solo los primeros registros.

In [0]:
#Contamos los registros del campo "value" que no son nulos, en caso de que existiesen:
df_bronze.filter(
    df_bronze.value.isNotNull()
).count()

2793

In [0]:
#Visualizamos los datos no nulos del campo "value"
display(
    df_bronze.filter(
        df_bronze.value.isNotNull()
    )
)

indicator,country,countryiso3code,date,value,unit,obs_status,decimal
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZI, Africa Western and Central)",AFW,2023,8917517.0,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZI, Africa Western and Central)",AFW,2022,8186999.0,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZI, Africa Western and Central)",AFW,2021,8567214.0,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZI, Africa Western and Central)",AFW,2020,8100808.0,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZI, Africa Western and Central)",AFW,2019,8562750.13,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZI, Africa Western and Central)",AFW,2018,7919906.59,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZI, Africa Western and Central)",AFW,2017,7328156.16,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZI, Africa Western and Central)",AFW,2016,6464269.34,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZI, Africa Western and Central)",AFW,2015,6722792.36,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(ZI, Africa Western and Central)",AFW,2014,6567432.71,,,0


# Resultado
El resultado obtenido confirma lo que sospechabamos previamente:
- La columna value sí contiene datos válidos.
- La API si contiene información en este campo.
- El problema está en cómo Spark infirió el esquema durante la creación del DataFrame.

#Conclusión
La columna value fue inferida incorrectamente por Spark como VOID; por lo que, en la capa silver; procederemos a corregir esta columna.

# 4. Capa Silver

Transformaciones:
- Limpieza
- Tipificación
- Eliminación de duplicados
- Validación de campos

In [0]:
df.printSchema()
df.show(5)

root
 |-- indicator: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- value: string (nullable = true)
 |-- country: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- value: string (nullable = true)
 |-- countryiso3code: string (nullable = true)
 |-- date: string (nullable = true)
 |-- value: void (nullable = true)
 |-- unit: string (nullable = true)
 |-- obs_status: string (nullable = true)
 |-- decimal: long (nullable = true)

+--------------------+--------------------+---------------+----+-----+----+----------+-------+
|           indicator|             country|countryiso3code|date|value|unit|obs_status|decimal|
+--------------------+--------------------+---------------+----+-----+----+----------+-------+
|{IS.SHP.GOOD.TU, ...|{ZH, Africa Easte...|            AFE|2025| NULL|    |          |      0|
|{IS.SHP.GOOD.TU, ...|{ZH, Africa Easte...|            AFE|2024| NULL|    |          |      0|
|{IS.SHP.GOOD.TU, ...|{ZH, Africa Easte...| 

# 5. Capa Gold

Construcción del modelo dimensional.
